# Supervised analysis (_Mushroom_)

In [7]:
from experiments.utils.constants import RANDOM_SEED, SOM_LEARNING_RATE_DECAY_FN

VERBOSE = True
DATASET_NAME = "Mushroom"
DATASET_ID = 73

EXPORT_MODE = False
EXPORT_DIR = "_exports"

print(DATASET_NAME)

Mushroom


## Dataset

In [8]:
# fetch dataset
from ucimlrepo import fetch_ucirepo
dataset = fetch_ucirepo(id=DATASET_ID)
X = dataset.data.features.values
y = dataset.data.targets.values.ravel()
print(f"Dataset shape: {X.shape}, {y.shape}")

Dataset shape: (8124, 22), (8124,)


In [9]:
# encode data
import pandas as pd
X = pd.get_dummies(pd.DataFrame(X), dummy_na=True).values

In [10]:
# scale data
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [11]:
# encode labels
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y_encoded = le.fit_transform(y)
label_decoder = {i: label for i, label in enumerate(le.classes_)}
num_unique_y = len(le.classes_)
print(f"Classes: {num_unique_y}")

Classes: 2


## Re-create model

In [12]:
from minisom_representation import calc_som_hyparams, SomRepresentation

In [13]:
# hyperparameters
recommended_params = calc_som_hyparams(X_scaled, initial_sigma_factor=3.0)
print("Recommended SOM parameters:", recommended_params)
d1, d2, sigma = map(recommended_params.get, ("d1", "d2", "sigma"))
decay_function = SOM_LEARNING_RATE_DECAY_FN
epoch = 20

Recommended SOM parameters: {'d1': 21, 'd2': 22, 'sigma': 7.33}


In [14]:
# fit SOM representation
som_rep = SomRepresentation(d1=d1, d2=d2, sigma=sigma, random_seed=RANDOM_SEED, verbose=VERBOSE, decay_function=decay_function) \
    .fit_online(X_scaled, num_iteration=epoch)

 [ 162480 / 162480 ] 100% - 0:00:00 left 
 quantization error: 6.56711584803875

 An SOM representation has been fitted as follows:
------------------------------------------------------- 

Fit strategy: online 

Hyperparameters of SOM: 

{'input_len': 138, 'x': 21, 'y': 22, 'sigma': 7.33, 'topology': 'rectangular', 'learning_rate': 0.5, 'decay_function': 'linear_decay_to_zero', 'sigma_decay_function': 'asymptotic_decay', 'neighborhood_function': 'gaussian', 'activation_distance': 'euclidean', 'random_seed': 42, 'num_iteration': 20, 'use_epochs': True, 'random_order': True, 'verbose': True} 

Quality of SOM: 

Quantization Error (QE):	6.56711584803875
Topographic Error (TE): 	0.00012309207287050715


## Inspection

In [15]:
from utils.plotting import PlotlyHelperArgs

In [16]:
# create Basin
from lilypond import Basin
basin = Basin.from_som_representation(som_rep, random_seed=RANDOM_SEED, verbose=VERBOSE)

In [17]:
# lilypond visual
basin.pond() \
    .rhizome_layer() \
    .pad_layer() \
    .petal_layer() \
    .visualize(width=800, height=600);

## Extra figures

In [18]:
plot_args = dict(
    **PlotlyHelperArgs.Figsize(w=1200, h=1200),
    **PlotlyHelperArgs.FullStretch,
    **PlotlyHelperArgs.HiddenTicks(d1=d1, d2=d2),
    font=dict(size=35),
    showlegend=False
)

In [19]:
import plotly.express as px
palette = ["#AB63FA", "crimson"]
marker_colors = [palette[val % len(palette)] for val in y_encoded]

In [20]:
fig1 = basin.pond() \
    .pad_layer(gap="nogap") \
    .rhizome_layer(min_width=10, max_width=30) \
    .attraction_layer(X_scaled, jitter_amount=.25, marker=dict(color="rgba(255,0,0,0%)", line=dict(color=marker_colors, width=5), symbol="star-diamond", size=30, opacity=.8), name="Projection of training data colored by class") \
    .visualize(**plot_args);

In [21]:
if EXPORT_MODE:
	fig1.write_image(EXPORT_DIR + "/03_03_lilypond_01.png")

---

### The below cells are not part of the experiment. They are used to persist the data and register the model in Databricks and Bianor for further interactive investigation.

---

## Preparation

In [22]:
MODEL_REGISTRATION_MODE = False
DATASET_PERSIST_MODE = True

In [23]:
import pandas as pd
import mlflow

from dotenv import load_dotenv
from utils.databricks_util import get_spark, get_catalog_path, CATALOG, SCHEMA

In [24]:
load_dotenv()
spark = get_spark()

## Persist data in Databricks

In [25]:
TABLE_NAME = f"T_{DATASET_NAME}".lower()
TABLE_PATH = get_catalog_path(TABLE_NAME)
print(TABLE_PATH)

workspace.lilypond_experiments.t_mushroom


In [26]:
if DATASET_PERSIST_MODE:
	# persist full dataset as a managed table
    spark.createDataFrame(
		pd.DataFrame(X) \
			.assign(label=y_encoded) \
			.reset_index(names="id") \
        	.rename(columns={"label": "class"})
	).write \
		.mode("overwrite") \
		.saveAsTable(TABLE_PATH)

In [27]:
data_dbdf = spark.read \
    .table(TABLE_PATH)
data_dbdf.show(5)

+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+-----+----+-----+-----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+-----+-----+-----+----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|  id|    0|    1|    2|    3|    4|    5|    6|    7|    8|    9|   10|   11|   12|   13|   14|   15|   16|   17|   18|   19|   20|   21|   22|   23|   24|   25|   2

In [28]:
# separate variables
primary_key = ['id']
target = ['class']
data_df = data_dbdf.toPandas()
features = data_df.columns.difference((primary_key + target), sort=False).tolist()
assert primary_key[0] not in features, "Primary key shall not be in Features"
assert all(t not in features for t in target), "Targets must not be in Features"
print("Primary key:", primary_key)
print("Targets:", target)
print("Features:", features)

Primary key: ['id']
Targets: ['class']
Features: ['0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12', '13', '14', '15', '16', '17', '18', '19', '20', '21', '22', '23', '24', '25', '26', '27', '28', '29', '30', '31', '32', '33', '34', '35', '36', '37', '38', '39', '40', '41', '42', '43', '44', '45', '46', '47', '48', '49', '50', '51', '52', '53', '54', '55', '56', '57', '58', '59', '60', '61', '62', '63', '64', '65', '66', '67', '68', '69', '70', '71', '72', '73', '74', '75', '76', '77', '78', '79', '80', '81', '82', '83', '84', '85', '86', '87', '88', '89', '90', '91', '92', '93', '94', '95', '96', '97', '98', '99', '100', '101', '102', '103', '104', '105', '106', '107', '108', '109', '110', '111', '112', '113', '114', '115', '116', '117', '118', '119', '120', '121', '122', '123', '124', '125', '126', '127', '128', '129', '130', '131', '132', '133', '134', '135', '136', '137']


In [29]:
# create feature dataframe
feature_dbdf = data_dbdf.select(primary_key + features)
feature_dbdf.show(5)

+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+-----+----+-----+-----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+----+-----+-----+-----+----+-----+-----+-----+----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+-----+
|  id|    0|    1|    2|    3|    4|    5|    6|    7|    8|    9|   10|   11|   12|   13|   14|   15|   16|   17|   18|   19|   20|   21|   22|   23|   24|   25|   26|   2

In [30]:
# create feature table
from databricks.feature_engineering import FeatureEngineeringClient
FEATURE_TABLE_NAME = f"{TABLE_NAME}_feature"
FEATURE_TABLE_PATH = get_catalog_path(FEATURE_TABLE_NAME)
print(FEATURE_TABLE_PATH)

workspace.lilypond_experiments.t_mushroom_feature


In [31]:
if DATASET_PERSIST_MODE:
	feClient = FeatureEngineeringClient()
	feClient.create_table(
		name=FEATURE_TABLE_PATH,
		primary_keys=primary_key,
		df=feature_dbdf,
		description=f"{DATASET_NAME} features (original)",
		tags={"source": "bronze", "format": "delta"}
	)

2026/09/16 01:14:13 INFO databricks.ml_features._compute_client._compute_client: Setting columns ['id'] of table 'workspace.lilypond_experiments.t_mushroom_feature' to NOT NULL.
2026/09/16 01:14:16 INFO databricks.ml_features._compute_client._compute_client: Setting Primary Keys constraint ['id'] on table 'workspace.lilypond_experiments.t_mushroom_feature'.
2026/09/16 01:14:32 INFO databricks.ml_features._compute_client._compute_client: Created feature table 'workspace.lilypond_experiments.t_mushroom_feature'.


## Register representation model in Databricks

In [ ]:
MODEL_NAME = f"som-{DATASET_NAME.lower()}"
MODEL_PATH = get_catalog_path(MODEL_NAME)
EXPERIMENT_NAME = f"/Users/matebalogh@ophelia-rnd.dev/Bianor_LilypondExperiments_{DATASET_NAME}"
print(MODEL_PATH, EXPERIMENT_NAME)

In [ ]:
if MODEL_REGISTRATION_MODE:

	mlflow.set_experiment(experiment_name=EXPERIMENT_NAME)

	class MLflowSomModelWrapper(mlflow.pyfunc.PythonModel):
		from typing import Any

		def __init__(self, model:SomRepresentation, scaler):
			self.model = model
			self.scaler = scaler
		def predict(self, context, model_input, params: dict[str, Any] | None = None):
			"""First transforms the input data via scaler, then predicts the winner node of the SOM."""
			return [self.model.som.winner(x) for x in self.scaler.transform(model_input.to_numpy())]

	with mlflow.start_run():
		model = MLflowSomModelWrapper(som_rep, scaler)

		mlflow.log_metric("QE", som_rep.quantization_error)
		mlflow.log_metric("TE", som_rep.topographic_error)

		mlflow.pyfunc.log_model(
			python_model=model,
			name=MODEL_NAME,
			input_example=pd.DataFrame(X_scaled[:3]),
			pip_requirements=[
				"numpy",
				"pandas",
				"scikit-learn==1.5.2",
				"mlflow",
				"minisom",
			],
			registered_model_name=MODEL_PATH
		)

else: print("Skipping MLflow model registration.")

In [ ]:
version = 1

registered_model = f"{MODEL_NAME}/{version}"
print(registered_model)

registered_model_location = get_catalog_path(registered_model)
print(registered_model_location)

## Register metadata in Bianor

In [32]:
from bianor_databricks_kit import BianorRecordManager
bianor_recorder = BianorRecordManager(catalog=CATALOG, schema=SCHEMA, spark=spark)

In [ ]:
if MODEL_REGISTRATION_MODE:

		# first registration
		# bianor_recorder.new_representation(name=f"{DATASET_NAME} Representation", som_model_location=registered_model_location, features_location="c.s.t")

		rep_id = "7fb4de84-e961-4a00-b48b-d428b4be6c13"

		# update existing
		bianor_recorder.update_representation(
			representation_id=rep_id,
			features_location=FEATURE_TABLE_PATH
		)

		# register projection layers

		class_names = le.classes_
		sample_locations = [f"V_{DATASET_NAME}_class_{cn}".lower() for cn in class_names]
		colors = palette
		names = [f"Training data - {cn}" for cn in class_names]

		for color, name, sample_loc in zip(colors, names, sample_locations):
			marker_dict = dict(
				color=color,
				symbol="star-diamond",
				opacity=0.9
			)

			proj_id = bianor_recorder.new_projection_layer(
				name=name,
				marker_dict=marker_dict,
				samples_location=get_catalog_path(sample_loc)
			)

			bianor_recorder.new_map_representation_projection_layer(representation_id=rep_id, projection_layer_id=proj_id)


else: print("Skipping Bianor metadata registration.")